**Step 3 of 3 — MongoDB Ingestion**

Transforms the enriched flat dataframe into nested MongoDB documents and bulk-inserts them into a MongoDB Atlas collection.

Each document follows the soft-schema structure:
```json
{
  "transaction_id": "txn_000001",
  "time": 3600,
  "amount": 149.62,
  "class": 0,
  "pca_features": { "V1": -1.359807, "V2": ..., "V28": ... },
  "cardholder": {
    "account_age_days": 730,
    "avg_monthly_spend": 842.50,
    "velocity_7d": 12
  },
  "merchant": {
    "mcc": "5411",
    "region": "Western Europe",
    "merchant_id": "merch_08821"
  }
}
```

**Run order:** ingest.ipynb → enrich.ipynb → load_mongo.ipynb

In [ ]:
#Install Dependencies
!pip install pymongo --quiet

In [ ]:
#Imports & Logging Setup
import logging
import os
import pandas as pd
from pymongo import MongoClient, errors, InsertOne
# Configure logging to both file and notebook output
logging.basicConfig(
    filename="load_mongo.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)
console = logging.StreamHandler()
console.setLevel(logging.INFO)
logging.getLogger("").addHandler(console)

In [ ]:
#connecting to DB
INPUT_PATH = "enriched.csv"
MONGO_URI = os.environ.get("MONGO_URI", "mongodb://localhost:27017/")

DB_NAME = "fraud_detection"
COLLECTION_NAME = "transactions"
BATCH_SIZE = 1000

PCA_COLS = [f"V{i}" for i in range(1, 29)]


In [ ]:
#Load Enriched Data
def load_enriched(path: content/enriched_data) -> pd.DataFrame:
    logging.info(f"Loading enriched data from {path}")
    try:
        df = pd.read_csv(path)
        logging.info(f"Loaded {len(df)} rows.")
        return df
    except FileNotFoundError as e:
        logging.error(f"File not found: {e}")
        raise

df = load_enriched(INPUT_PATH)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
#Define Document Transformation
#Convert a single enriched row into the nested MongoDB document structure
def row_to_document(row: pd.Series) -> dict:
    return {
        "transaction_id": row["transaction_id"],
        "time": int(row["Time"]),
        "amount": round(float(row["Amount"]), 2),
        "class": int(row["Class"]),
        "pca_features": {
            col: round(float(row[col]), 6) for col in PCA_COLS
        },
        "cardholder": {
            "account_age_days": int(row["cardholder_account_age_days"]),
            "avg_monthly_spend": round(float(row["cardholder_avg_monthly_spend"]), 2),
            "velocity_7d": int(row["cardholder_velocity_7d"])
        },
        "merchant": {
            "mcc": str(row["merchant_mcc"]),
            "region": str(row["merchant_region"]),
            "merchant_id": str(row["merchant_id"])
        }
    }

In [ ]:
#Connect to MongoDB Atlas
logging.info(f"Connecting to MongoDB...")
try:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    client.server_info()  # Trigger connection to catch auth errors early
    logging.info("Connected successfully.")
    print("Connected to MongoDB successfully.")
except errors.ServerSelectionTimeoutError as e:
    logging.error(f"Could not connect to MongoDB: {e}")
    raise

db = client[DB_NAME]
collection = db[COLLECTION_NAME]

In [ ]:
#Drop Existing Collection
collection.drop()
logging.info(f"Dropped existing '{COLLECTION_NAME}' collection.")
print(f"Dropped '{COLLECTION_NAME}' collection.")

In [ ]:
#Convert Rows to Documents
logging.info("Converting rows to documents...")
documents = [row_to_document(row) for _, row in df.iterrows()]
logging.info(f"{len(documents)} documents prepared.")
print(f"{len(documents)} documents ready for insertion.")

In [ ]:
#Bulk Insert in Batches
def bulk_insert(collection, documents: list) -> int:
    """Insert a list of documents in bulk. Returns the number inserted."""
    try:
        result = collection.bulk_write([InsertOne(doc) for doc in documents])
        return result.inserted_count
    except errors.BulkWriteError as e:
        logging.error(f"Bulk write error: {e.details}")
        raise

total_inserted = 0
num_batches = (len(documents) + BATCH_SIZE - 1) // BATCH_SIZE

for i in range(0, len(documents), BATCH_SIZE):
    batch = documents[i: i + BATCH_SIZE]
    inserted = bulk_insert(collection, batch)
    total_inserted += inserted
    batch_num = i // BATCH_SIZE + 1
    logging.info(f"Inserted batch {batch_num}/{num_batches}: {inserted} documents.")
    if batch_num % 50 == 0 or batch_num == num_batches:
        print(f"  Batch {batch_num}/{num_batches} — {total_inserted} total inserted")

logging.info(f"Load complete. Total documents inserted: {total_inserted}")

In [ ]:
#Close Connection
client.close()
logging.info("MongoDB connection closed.")
print("Connection closed. Pipeline complete.")